# [Kaggle] 06 - Evaluate All Models (Paper Format)

Cháº¡y tá»«ng cell Ä‘á»ƒ khoanh vÃ¹ng lá»—i dá»… hÆ¡n. PhoBERT Ä‘Æ°á»£c Ä‘á»ƒ riÃªng á»Ÿ cell cuá»‘i vÃ¬ phá»¥ thuá»™c `transformers` vÃ  mÃ´i trÆ°á»ng cá»§a báº¡n.

In [1]:
import json
import os
import pickle
import sys
from collections import defaultdict
from pathlib import Path

os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader

def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'train.jsonl').exists() and (candidate / 'models').exists() and (candidate / 'preprocessed').exists():
            return candidate
    raise FileNotFoundError('Could not locate repo root.')

ROOT_DIR = find_repo_root()
RESEARCH_DIR = ROOT_DIR / 'research_pipeline'
DATA_DIR = ROOT_DIR / 'data'
MODELS_DIR = ROOT_DIR / 'models'
PREPROCESSED_DIR = ROOT_DIR / 'preprocessed'
if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('ROOT_DIR:', ROOT_DIR)

ASPECTS = ['CAMERA', 'FEATURES', 'PERFORMANCE', 'DESIGN', 'PRICE', 'GENERAL', 'SCREEN', 'BATTERY', 'STORAGE', 'SER&ACC']
SENTIMENTS = ['POSITIVE', 'NEUTRAL', 'NEGATIVE']
LABEL_NAMES = [f'{a}#{s}' for a in ASPECTS for s in SENTIMENTS]

from src.utils.preprocess import load_raw_data, tokenize_baseline
from src.utils.metrics import bio_tags_to_spans, evaluate_spans_f1, evaluate_asc
from src.utils.engine import predict_ate, predict_asc
from src.ate.ate_dataset import ATEDataset, BIO_TAGS as BIO_TAGS_ATE
from src.ate.ate_model import build_ate_model
from src.asc.asc_dataset import ASCDataset
from src.asc.asc_model import build_asc_model
from src.e2e.e2e_baseline_dataset import E2EBaselineDataset, BIO_TAGS as BIO_TAGS_E2E_BASE
from src.e2e.e2e_dataset import E2EDataset, BIO_TAGS as BIO_TAGS_E2E, NUM_TAGS as NUM_TAGS_E2E

with open(PREPROCESSED_DIR / 'word2idx.pkl', 'rb') as f:
    word2idx = pickle.load(f)
emb_matrix = np.load(PREPROCESSED_DIR / 'emb_matrix.npy')
with open(PREPROCESSED_DIR / 'test_segmented.json', 'r', encoding='utf-8') as f:
    test_items_seg = json.load(f)
test_items_raw = load_raw_data(DATA_DIR / 'test.jsonl')

print('Loaded vocab:', len(word2idx))
print('Embeddings:', emb_matrix.shape)
print('Test items:', len(test_items_raw))

MAX_LEN_ATE = 128
MAX_LEN_ASC = 128
MAX_LEN_PHOBERT = 256

def load_state_dict_if_exists(model, path):
    if not path.exists():
        print('Missing model file:', path)
        return None
    model.load_state_dict(torch.load(path, map_location=device))
    model.to(device)
    model.eval()
    return model

def evaluate_absa_paper_format(pred_spans_list, true_spans_list, model_name='Model'):
    metrics = {'Aspect': {'tp': 0, 'fp': 0, 'fn': 0}, 'Polarity': {'tp': 0, 'fp': 0, 'fn': 0}, 'Aspect-Polarity': {'tp': 0, 'fp': 0, 'fn': 0}}
    aspect_counts = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})
    sentiment_counts = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})
    aspect_sentiment_counts = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})
    for true_spans, pred_spans in zip(true_spans_list, pred_spans_list):
        t_ap = set((l, s, e) for l, s, e in true_spans)
        p_ap = set((l, s, e) for l, s, e in pred_spans)
        t_a = set((l.split('#')[0], s, e) for l, s, e in true_spans)
        p_a = set((l.split('#')[0], s, e) for l, s, e in pred_spans)
        t_p = set(((l.split('#')[1] if '#' in l else 'UNK'), s, e) for l, s, e in true_spans)
        p_p = set(((l.split('#')[1] if '#' in l else 'UNK'), s, e) for l, s, e in pred_spans)
        def update(true_set, pred_set, overall, cls):
            for span in true_set:
                key = span[0]
                if span in pred_set:
                    overall['tp'] += 1; cls[key]['tp'] += 1
                else:
                    overall['fn'] += 1; cls[key]['fn'] += 1
            for span in pred_set:
                if span not in true_set:
                    key = span[0]
                    overall['fp'] += 1; cls[key]['fp'] += 1
        update(t_ap, p_ap, metrics['Aspect-Polarity'], aspect_sentiment_counts)
        update(t_a, p_a, metrics['Aspect'], aspect_counts)
        update(t_p, p_p, metrics['Polarity'], sentiment_counts)
    def calc(tp, fp, fn):
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        return p * 100, r * 100, f * 100
    def macro(counts, keys):
        vals = [calc(counts[k]['tp'], counts[k]['fp'], counts[k]['fn']) for k in keys]
        return tuple(np.mean([v[i] for v in vals]) for i in range(3))
    rows = []
    for task in ['Aspect', 'Polarity', 'Aspect-Polarity']:
        c = metrics[task]
        p_micro, r_micro, f1_micro = calc(c['tp'], c['fp'], c['fn'])
        if task == 'Aspect':
            keys, counts = ASPECTS, aspect_counts
        elif task == 'Polarity':
            keys, counts = SENTIMENTS, sentiment_counts
        else:
            keys = [f'{a}#{s}' for a in ASPECTS for s in SENTIMENTS]
            counts = aspect_sentiment_counts
        p_macro, r_macro, f1_macro = macro(counts, keys)
        rows.append({'System': task, 'PMicro': p_micro, 'RMicro': r_micro, 'F1Micro': f1_micro, 'PMacro': p_macro, 'RMacro': r_macro, 'F1Macro': f1_macro})
    df = pd.DataFrame(rows).round(2).set_index('System')
    print(f'\n{"=" * 88}\n{model_name}\n{"=" * 88}')
    display(df)
    return df

def evaluate_ate_spans(pred_spans, true_spans, model_name='Model'):
    res = evaluate_spans_f1(pred_spans, true_spans)
    df = pd.DataFrame([{'Model': model_name, 'Precision': res['precision'] * 100, 'Recall': res['recall'] * 100, 'F1': res['f1'] * 100}]).round(2).set_index('Model')
    print(f'\n{"=" * 88}\n{model_name}\n{"=" * 88}')
    display(df)
    return df

def evaluate_asc_predictions(y_true, y_pred, model_name='Model'):
    res = evaluate_asc(y_true, y_pred, class_names=['POSITIVE', 'NEGATIVE', 'NEUTRAL'])
    df = pd.DataFrame([{'Model': model_name, 'Accuracy': res['accuracy'] * 100, 'Macro_F1': res['macro_f1'] * 100, 'Weighted_F1': res['weighted_f1'] * 100}]).round(2).set_index('Model')
    print(f'\n{"=" * 88}\n{model_name}\n{"=" * 88}')
    display(df)
    return df

def segmented_raw_labels_to_word_spans(seg_text, raw_labels, max_len):
    words = seg_text.split()[:max_len]
    positions, pos = [], 0
    for w in words:
        w_len = len(w.replace('_', ' '))
        positions.append((pos, pos + w_len))
        pos += w_len + 1
    spans = []
    for start_char, end_char, label in raw_labels:
        idxs = [i for i, (s, e) in enumerate(positions) if s < end_char and e > start_char]
        if idxs:
            spans.append((label, idxs[0], idxs[-1] + 1))
    return spans

def pipeline_predict(seg_text, ate_model, asc_model, word2idx, device, max_len=128):
    words = seg_text.split()[:max_len]
    if not words:
        return []
    seq, length = tokenize_baseline(seg_text, word2idx, max_len)
    seq_tensor = torch.tensor([seq], dtype=torch.long, device=device)
    len_tensor = torch.tensor([length], dtype=torch.long)
    mask = torch.zeros((1, max_len), dtype=torch.bool, device=device)
    mask[:, :length] = True
    with torch.no_grad():
        pred_tags = ate_model(seq_tensor, mask=mask, lens=len_tensor)[0]
    aspect_spans = bio_tags_to_spans(pred_tags, BIO_TAGS_ATE, length)
    results = []
    for aspect, start, end in aspect_spans:
        marked_text = ' '.join(words[:start] + ['[ASP]'] + words[start:end] + ['[ASP]'] + words[end:])
        asc_seq, asc_len = tokenize_baseline(marked_text, word2idx, max_len)
        asc_seq_tensor = torch.tensor([asc_seq], dtype=torch.long, device=device)
        asc_len_tensor = torch.tensor([asc_len], dtype=torch.long)
        with torch.no_grad():
            logits = asc_model(asc_seq_tensor, asc_len_tensor)
        sentiment = {0: 'POSITIVE', 1: 'NEGATIVE', 2: 'NEUTRAL'}[logits.argmax(dim=1).item()]
        results.append((f'{aspect}#{sentiment}', start, end))
    return results

print('Setup done.')

Device: cuda
ROOT_DIR: C:\Users\Dell\machine learning\data NLP vietnamese\data\data\Vietnamese-Aspect-based-sentiment-analyst-project


c:\Users\Dell\anaconda3\envs\my_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[e2e_dataset] Using underthesea for segmentation
Loaded vocab: 6169
Embeddings: (6169, 150)
Test items: 2225
Setup done.


In [2]:
print('Preparing datasets for E2E BiGRU-CRF...')
test_ds_bigru = E2EBaselineDataset(test_items_seg, word2idx, MAX_LEN_ATE)
test_loader_bigru = DataLoader(test_ds_bigru, batch_size=64, shuffle=False)
e2e_bigru_model = build_ate_model('BiGRU', len(word2idx), 150, 256, len(BIO_TAGS_E2E_BASE), pretrained_emb=emb_matrix, n_layers=2, dropout=0.3)
if load_state_dict_if_exists(e2e_bigru_model, MODELS_DIR / 'best_e2e_baseline_BiGRU-CRF.pt') is not None:
    test_res_bigru = predict_ate(e2e_bigru_model, test_loader_bigru, device, bio_tags_list=BIO_TAGS_E2E_BASE)
    pred_spans_bigru = [bio_tags_to_spans(pt, BIO_TAGS_E2E_BASE, l) for pt, l in zip(test_res_bigru['pred_tags'], test_res_bigru['lengths'])]
    true_spans_bigru = [bio_tags_to_spans(tt, BIO_TAGS_E2E_BASE, l) for tt, l in zip(test_res_bigru['true_tags'], test_res_bigru['lengths'])]
    e2e_bigru_df = evaluate_absa_paper_format(pred_spans_bigru, true_spans_bigru, 'E2E BiGRU-CRF')
else:
    e2e_bigru_df = None

Preparing datasets for E2E BiGRU-CRF...


RuntimeError: Error(s) in loading state_dict for ATESequenceCRF:
	size mismatch for emb.weight: copying a param with shape torch.Size([6168, 150]) from checkpoint, the shape in current model is torch.Size([6169, 150]).

In [ ]:
print('Preparing datasets for ATE BiLSTM-CRF...')
test_ds_ate = ATEDataset(test_items_seg, word2idx, MAX_LEN_ATE)
test_loader_ate = DataLoader(test_ds_ate, batch_size=64, shuffle=False)
ate_model = build_ate_model('BiLSTM', len(word2idx), 150, 256, len(BIO_TAGS_ATE), pretrained_emb=emb_matrix, n_layers=2, dropout=0.3)
if load_state_dict_if_exists(ate_model, MODELS_DIR / 'best_ate_BiLSTM-CRF.pt') is not None:
    test_res_ate = predict_ate(ate_model, test_loader_ate, device, bio_tags_list=BIO_TAGS_ATE)
    pred_spans_ate = [bio_tags_to_spans(pt, BIO_TAGS_ATE, l) for pt, l in zip(test_res_ate['pred_tags'], test_res_ate['lengths'])]
    true_spans_ate = [bio_tags_to_spans(tt, BIO_TAGS_ATE, l) for tt, l in zip(test_res_ate['true_tags'], test_res_ate['lengths'])]
    ate_df = evaluate_ate_spans(pred_spans_ate, true_spans_ate, 'ATE BiLSTM-CRF')
else:
    ate_df = None

Preparing datasets for ATE BiLSTM-CRF...

ATE BiLSTM-CRF


,Precision,Recall,F1
Model,,,
ATE BiLSTM-CRF,41.41,36.7,38.91


In [ ]:
print('Preparing datasets for ASC BiGRU...')
test_ds_asc = ASCDataset(test_items_seg, word2idx, MAX_LEN_ASC)
test_loader_asc = DataLoader(test_ds_asc, batch_size=128, shuffle=False)
asc_model = build_asc_model('BiGRU', len(word2idx), 150, 256, num_classes=3, pretrained_emb=emb_matrix, n_layers=2, dropout=0.3)
if load_state_dict_if_exists(asc_model, MODELS_DIR / 'best_asc_BiGRU.pt') is not None:
    criterion = torch.nn.CrossEntropyLoss()
    test_res_asc = predict_asc(asc_model, test_loader_asc, criterion, device)
    asc_df = evaluate_asc_predictions(test_res_asc['labels'], test_res_asc['preds'], 'ASC BiGRU')
else:
    asc_df = None

Preparing datasets for ASC BiGRU...

ASC BiGRU


,Accuracy,Macro_F1,Weighted_F1
Model,,,
ASC BiGRU,90.25,80.22,90.74


In [ ]:
print('Preparing pipeline evaluation (ATE + ASC)...')
pred_spans_pipe, true_spans_pipe = [], []
for raw_item, seg_item in zip(test_items_raw, test_items_seg):
    pred_spans_pipe.append(pipeline_predict(seg_item['text'], ate_model, asc_model, word2idx, device, max_len=MAX_LEN_ASC))
    true_spans_pipe.append(segmented_raw_labels_to_word_spans(seg_item['text'], raw_item['labels'], MAX_LEN_ASC))
pipeline_df = evaluate_absa_paper_format(pred_spans_pipe, true_spans_pipe, 'Pipeline (ATE BiLSTM + ASC BiGRU)')

Preparing pipeline evaluation (ATE + ASC)...

Pipeline (ATE BiLSTM + ASC BiGRU)


,PMicro,RMicro,F1Micro,PMacro,RMacro,F1Macro
System,,,,,,
Aspect,41.97,36.96,39.31,37.69,32.91,35.11
Polarity,40.99,36.10,38.39,32.44,30.41,30.95
Aspect-Polarity,39.37,34.67,36.87,26.71,24.00,24.85


In [ ]:
print('Preparing PhoBERT E2E evaluation...')
try:
    from src.e2e.e2e_model import E2EPhoBertCRF
    test_ds_phobert = E2EDataset(test_items_raw, max_len=MAX_LEN_PHOBERT)
    test_loader_phobert = DataLoader(test_ds_phobert, batch_size=16, shuffle=False)
    phobert_model = E2EPhoBertCRF(num_unified_tags=NUM_TAGS_E2E, dropout=0.3)
    if load_state_dict_if_exists(phobert_model, MODELS_DIR / 'best_e2e_phobert.pt') is not None:
        from src.utils.engine import predict_e2e
        test_res_phobert = predict_e2e(phobert_model, test_loader_phobert, device)
        pred_spans_phobert = [bio_tags_to_spans(pt, BIO_TAGS_E2E, l) for pt, l in zip(test_res_phobert['pred_tags'], test_res_phobert['lengths'])]
        true_spans_phobert = [bio_tags_to_spans(tt, BIO_TAGS_E2E, l) for tt, l in zip(test_res_phobert['true_tags'], test_res_phobert['lengths'])]
        phobert_df = evaluate_absa_paper_format(pred_spans_phobert, true_spans_phobert, 'E2E PhoBERT-CRF')
    else:
        phobert_df = None
except Exception as exc:
    print('PhoBERT skipped:', exc)
    phobert_df = None

Preparing PhoBERT E2E evaluation...


c:\Users\Dell\anaconda3\envs\my_env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dell\.cache\huggingface\hub\models--vinai--phobert-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Tokenizing 2225 items...


Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


KeyboardInterrupt: 

In [ ]:
summary = []
if 'e2e_bigru_df' in globals() and e2e_bigru_df is not None:
    row = e2e_bigru_df.loc['Aspect-Polarity']
    summary.append({'Model': 'E2E BiGRU-CRF', 'Aspect-Polarity F1Micro': row['F1Micro'], 'Aspect-Polarity F1Macro': row['F1Macro']})
if 'ate_df' in globals() and ate_df is not None:
    row = ate_df.iloc[0]
    summary.append({'Model': 'ATE BiLSTM-CRF', 'ATE F1': row['F1']})
if 'asc_df' in globals() and asc_df is not None:
    row = asc_df.iloc[0]
    summary.append({'Model': 'ASC BiGRU', 'ASC Accuracy': row['Accuracy'], 'ASC Macro F1': row['Macro_F1']})
if 'pipeline_df' in globals() and pipeline_df is not None:
    row = pipeline_df.loc['Aspect-Polarity']
    summary.append({'Model': 'Pipeline (ATE BiLSTM + ASC BiGRU)', 'Aspect-Polarity F1Micro': row['F1Micro'], 'Aspect-Polarity F1Macro': row['F1Macro']})
if 'phobert_df' in globals() and phobert_df is not None:
    row = phobert_df.loc['Aspect-Polarity']
    summary.append({'Model': 'E2E PhoBERT-CRF', 'Aspect-Polarity F1Micro': row['F1Micro'], 'Aspect-Polarity F1Macro': row['F1Macro']})
summary_df = pd.DataFrame(summary).round(2)
if not summary_df.empty:
    display(summary_df)
else:
    print('No results were produced.')

,Model,ATE F1,ASC Accuracy,ASC Macro F1,Aspect-Polarity F1Micro,Aspect-Polarity F1Macro
0,ATE BiLSTM-CRF,38.91,NaN,NaN,NaN,NaN
1,ASC BiGRU,NaN,90.25,80.22,NaN,NaN
2,Pipeline (ATE BiLSTM + ASC BiGRU),NaN,NaN,NaN,36.87,24.85
